# Phase 3: Model Training & Evaluation - Isolation Forest

## Objective
Train an **Isolation Forest** classifier for anomaly-based timestomping detection and compare with Random Forest performance.

## Isolation Forest Approach:
- **Unsupervised Anomaly Detection**: Isolates anomalies by randomly selecting features and split values
- **Contamination Parameter**: Set based on known timestomped ratio (~0.032%)
- **No SMOTE Required**: Works directly with imbalanced data
- **Forensic Hypothesis**: Timestomped events exhibit anomalous patterns distinct from benign files

## Expected Behavior:
- Isolation Forest treats timestomped events as **outliers/anomalies**
- Should identify unusual patterns in temporal, path, and cross-artifact features
- May perform differently than Random Forest due to unsupervised nature

## Input
- **Engineered Dataset:** `data/processed/Phase 2 - Feature Engineering/features_engineered.csv`
- **Records:** 778,692 events
- **Features:** 87 columns (75 after exclusions)
- **Labels:** 247 timestomped events (1:3,151 imbalance)

## Output
- **Trained Model:** Isolation Forest classifier
- **Full Evaluation:** Same metrics as Random Forest for direct comparison
- **Saved to:** `data/processed/Phase 3 - Model Training/isolation_forest/`

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import joblib

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    roc_curve,
    precision_recall_curve, 
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 8)

print("✓ Libraries imported successfully")

In [ ]:
# Define paths
notebook_dir = Path.cwd()
print(f"Current working directory: {notebook_dir}")

# Navigate to project root
if 'notebooks' in str(notebook_dir):
    BASE_DIR = notebook_dir.parent.parent / 'data'
else:
    BASE_DIR = Path('data')

INPUT_FILE = BASE_DIR / 'processed' / 'Phase 2 - Feature Engineering' / 'features_engineered.csv'
OUTPUT_DIR = BASE_DIR / 'processed' / 'Phase 3 - Model Training' / 'isolation_forest'

# Ensure output directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Directory Configuration:")
print(f"  Input:  {INPUT_FILE} {'✓' if INPUT_FILE.exists() else '✗ NOT FOUND'}")
print(f"  Output: {OUTPUT_DIR} ✓")

---
## 2. Load Data & Initial Exploration

In [ ]:
print("\n" + "=" * 80)
print("LOADING ENGINEERED FEATURES")
print("=" * 80)

# Load dataset
df = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')

print(f"\n📊 Dataset Loaded:")
print(f"   Records: {len(df):,}")
print(f"   Features: {len(df.columns)}")
print(f"   Timestomped events: {df['is_timestomped'].sum()}")

# Class distribution
print(f"\n📈 Class Distribution:")
class_counts = df['is_timestomped'].value_counts().sort_index()
for label, count in class_counts.items():
    pct = count / len(df) * 100
    print(f"   Class {int(label)}: {count:,} ({pct:.3f}%)")

imbalance_ratio = class_counts[0] / class_counts[1]
print(f"\n   ⚠️  Imbalance Ratio: 1:{int(imbalance_ratio)}")

# Memory usage
print(f"\n💾 Memory Usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

---
## 3. Data Preparation: Feature Selection & Type Handling

In [ ]:
print("\n" + "=" * 80)
print("DATA PREPARATION")
print("=" * 80)

# Columns to exclude from features (prevent data leakage)
cols_to_exclude = [
    'is_timestomped',
    'is_timestomped_lf',
    'is_timestomped_usn',
    'timestomp_tool_executed',
    'timestomp_tool_executed_lf',
    'timestomp_tool_executed_usn',
    'case_id',
    'eventtime_dt',
    'label_source_both',
    'label_source_logfile',
    'label_source_usnjrnl',
    'label_source_nan',
]

# Get feature columns
feature_cols = [col for col in df.columns if col not in cols_to_exclude]

print(f"\n✓ Feature columns: {len(feature_cols)}")

# Convert bool to int
bool_cols = df[feature_cols].select_dtypes(include='bool').columns.tolist()
if bool_cols:
    for col in bool_cols:
        df[col] = df[col].astype(int)

# Create X and y
X = df[feature_cols].copy()
y = df['is_timestomped'].copy()
case_ids = df['case_id'].copy()

# Fill missing values
X = X.fillna(0)

print(f"\n✅ Data preparation complete!")
print(f"   Final X shape: {X.shape}")
print(f"   Features: {X.shape[1]}")
print(f"   Samples: {X.shape[0]:,}")

---
## 4. Train/Test Split: Case-Based Stratification

**Using same split strategy as Random Forest for fair comparison**

In [ ]:
print("\n" + "=" * 80)
print("TRAIN/TEST SPLIT: CASE-BASED STRATIFICATION")
print("=" * 80)

# Analyze case distribution
case_stats = df.groupby('case_id').agg({
    'is_timestomped': ['sum', 'count']
}).reset_index()
case_stats.columns = ['case_id', 'timestomped_count', 'total_events']

print(f"\n   Case Statistics:")
print(case_stats.to_string(index=False))

# Categorize cases
high_timestomp_cases = case_stats[case_stats['timestomped_count'] >= 30]['case_id'].tolist()
med_timestomp_cases = case_stats[(case_stats['timestomped_count'] >= 2) & (case_stats['timestomped_count'] < 30)]['case_id'].tolist()
low_timestomp_cases = case_stats[(case_stats['timestomped_count'] > 0) & (case_stats['timestomped_count'] < 2)]['case_id'].tolist()

# Manual stratified split
from sklearn.model_selection import train_test_split as split_list

train_cases = []
test_cases = []

if len(high_timestomp_cases) >= 2:
    train_high, test_high = split_list(high_timestomp_cases, test_size=0.25, random_state=42)
    train_cases.extend(train_high)
    test_cases.extend(test_high)
if len(med_timestomp_cases) >= 2:
    train_med, test_med = split_list(med_timestomp_cases, test_size=0.25, random_state=42)
    train_cases.extend(train_med)
    test_cases.extend(test_med)
if len(low_timestomp_cases) >= 2:
    train_low, test_low = split_list(low_timestomp_cases, test_size=0.25, random_state=42)
    train_cases.extend(train_low)
    test_cases.extend(test_low)

# Create train/test masks
train_mask = case_ids.isin(train_cases)
test_mask = case_ids.isin(test_cases)

# Split data
X_train = X[train_mask].copy()
X_test = X[test_mask].copy()
y_train = y[train_mask].copy()
y_test = y[test_mask].copy()

print(f"\n✅ Split complete!")
print(f"   Train: {len(X_train):,} samples, {int(y_train.sum())} timestomped")
print(f"   Test:  {len(X_test):,} samples, {int(y_test.sum())} timestomped")

---
## 5. Feature Scaling for Isolation Forest

**Isolation Forest is sensitive to feature scales**
- Standardizing features ensures equal contribution to anomaly scoring
- Unlike Random Forest, tree-based isolation benefits from normalized features

In [ ]:
print("\n" + "=" * 80)
print("FEATURE SCALING")
print("=" * 80)

print(f"\n1️⃣ Applying StandardScaler to features...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Feature scaling complete!")
print(f"   Train mean: {X_train_scaled.mean():.6f} (should be ~0)")
print(f"   Train std:  {X_train_scaled.std():.6f} (should be ~1)")
print(f"   Test mean:  {X_test_scaled.mean():.6f}")
print(f"   Test std:   {X_test_scaled.std():.6f}")

---
## 6. Model Training: Isolation Forest

**Model Configuration:**
- `contamination='auto'`: Automatically determined (can adjust based on known timestomp rate)
- `n_estimators=100`: Number of isolation trees
- `max_samples=256`: Subsample size for training each tree
- `max_features=1.0`: Use all features
- `random_state=42`: Reproducibility

**Contamination Parameter:**
- Expected contamination: ~0.03% (based on timestomp rate)
- Will test with 'auto' first, then experiment with manual values

In [ ]:
print("\n" + "=" * 80)
print("MODEL TRAINING: ISOLATION FOREST")
print("=" * 80)

# Calculate contamination rate from training data
contamination_rate = y_train.sum() / len(y_train)
print(f"\n1️⃣ Contamination Analysis:")
print(f"   Training set timestomped: {int(y_train.sum())} / {len(y_train):,}")
print(f"   Contamination rate: {contamination_rate:.6f} ({contamination_rate*100:.3f}%)")

# Try multiple contamination values
contamination_values = [
    contamination_rate,  # Exact rate
    0.001,  # 0.1% (more conservative)
    0.0005,  # 0.05% (very conservative)
]

print(f"\n2️⃣ Testing different contamination values...")
print(f"   Will evaluate: {[f'{c*100:.3f}%' for c in contamination_values]}")

# Train with exact contamination rate first
print(f"\n3️⃣ Training Isolation Forest with contamination={contamination_rate:.6f}...")

iso_forest = IsolationForest(
    n_estimators=100,
    max_samples=256,
    contamination=contamination_rate,
    max_features=1.0,
    bootstrap=False,
    n_jobs=-1,
    random_state=42,
    verbose=0
)

print(f"\n   Model Configuration:")
print(f"     - Trees: {iso_forest.n_estimators}")
print(f"     - Max Samples: {iso_forest.max_samples}")
print(f"     - Contamination: {contamination_rate:.6f} ({contamination_rate*100:.3f}%)")
print(f"     - Max Features: {iso_forest.max_features}")

# Train model
print(f"\n   Training on {len(X_train_scaled):,} samples...")
iso_forest.fit(X_train_scaled)

print(f"\n✅ Model training complete!")

# Training set predictions
# Isolation Forest returns -1 for outliers, 1 for inliers
# We need to convert: -1 (outlier/anomaly) -> 1 (timestomped), 1 (inlier) -> 0 (benign)
y_train_pred_raw = iso_forest.predict(X_train_scaled)
y_train_pred = np.where(y_train_pred_raw == -1, 1, 0)

# Get anomaly scores (lower = more anomalous)
y_train_scores = iso_forest.score_samples(X_train_scaled)

train_precision = precision_score(y_train, y_train_pred, zero_division=0)
train_recall = recall_score(y_train, y_train_pred, zero_division=0)
train_f1 = f1_score(y_train, y_train_pred, zero_division=0)

print(f"\n4️⃣ Training Set Performance:")
print(f"   Precision: {train_precision:.3f}")
print(f"   Recall: {train_recall:.3f}")
print(f"   F1-Score: {train_f1:.3f}")
print(f"\n   Anomaly scores range: [{y_train_scores.min():.3f}, {y_train_scores.max():.3f}]")
print(f"   Lower scores indicate more anomalous samples")

---
## 7. Model Evaluation: Test Set Performance

In [ ]:
print("\n" + "=" * 80)
print("MODEL EVALUATION: TEST SET")
print("=" * 80)

# Make predictions
print(f"\n1️⃣ Generating predictions on test set...")
y_test_pred_raw = iso_forest.predict(X_test_scaled)
y_test_pred = np.where(y_test_pred_raw == -1, 1, 0)

# Get anomaly scores (convert to probabilities for ROC/PR curves)
# Lower score = more anomalous, so we negate to get higher values for anomalies
y_test_scores = iso_forest.score_samples(X_test_scaled)
y_test_pred_proba = -y_test_scores  # Negate so higher = more anomalous

# Normalize to [0, 1] range for consistency with Random Forest
y_test_pred_proba = (y_test_pred_proba - y_test_pred_proba.min()) / (y_test_pred_proba.max() - y_test_pred_proba.min())

# Classification metrics
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)

print(f"\n2️⃣ Classification Metrics:")
print(f"   Precision: {test_precision:.3f} ({test_precision*100:.1f}%)")
print(f"   Recall:    {test_recall:.3f} ({test_recall*100:.1f}%)")
print(f"   F1-Score:  {test_f1:.3f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()

print(f"\n3️⃣ Confusion Matrix:")
print(f"\n                 Predicted")
print(f"                 Benign  Timestomped")
print(f"   Actual Benign    {tn:6d}     {fp:6d}")
print(f"   Actual Timestomped {fn:6d}     {tp:6d}")

print(f"\n   True Positives (TP):  {tp:,} ✓ (Correctly identified timestomped)")
print(f"   False Positives (FP): {fp:,} ✗ (Benign flagged as timestomped)")
print(f"   False Negatives (FN): {fn:,} ✗ (Timestomped missed)")
print(f"   True Negatives (TN):  {tn:,} ✓ (Correctly identified benign)")

# AUC scores
if y_test.sum() > 0:
    auc_roc = roc_auc_score(y_test, y_test_pred_proba)
    auc_pr = average_precision_score(y_test, y_test_pred_proba)
    
    print(f"\n4️⃣ AUC Scores:")
    print(f"   AUC-ROC: {auc_roc:.3f} (ability to rank predictions)")
    print(f"   AUC-PR:  {auc_pr:.3f} (precision-recall balance)")

# Forensic interpretation
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
total_flagged = tp + fp

print(f"\n5️⃣ Forensic Context Interpretation:")
print(f"   False Positive Rate: {fpr:.4f} ({fpr*100:.3f}%)")
print(f"   False Negative Rate: {fnr:.4f} ({fnr*100:.1f}%)")
print(f"\n   📊 Triage Effectiveness:")
print(f"      - Files to review: {total_flagged:,} out of {len(y_test):,}")
print(f"      - Investigation reduction: {(1 - total_flagged/len(y_test))*100:.2f}%")
print(f"      - Detection rate: {tp}/{int(y_test.sum())} timestomped files ({test_recall*100:.1f}%)")

print(f"\n✅ Evaluation complete!")

---
## 8. Comparison with Random Forest (v3 Final Model)

Load Random Forest metrics for direct comparison

In [ ]:
print("\n" + "=" * 80)
print("COMPARISON WITH RANDOM FOREST")
print("=" * 80)

# Load Random Forest metrics
rf_metrics_path = BASE_DIR / 'processed' / 'Phase 3 - Model Training' / 'v3_final' / 'evaluation_metrics.csv'

if rf_metrics_path.exists():
    rf_metrics = pd.read_csv(rf_metrics_path)
    
    print(f"\n📊 Model Comparison:")
    print(f"\n{'Metric':<25} {'Random Forest':>15} {'Isolation Forest':>18} {'Difference':>12}")
    print(f"{'-'*25} {'-'*15} {'-'*18} {'-'*12}")
    
    metrics_to_compare = [
        ('Precision', rf_metrics['test_precision'].values[0], test_precision),
        ('Recall', rf_metrics['test_recall'].values[0], test_recall),
        ('F1-Score', rf_metrics['test_f1'].values[0], test_f1),
        ('AUC-ROC', rf_metrics['auc_roc'].values[0], auc_roc if y_test.sum() > 0 else 0),
        ('AUC-PR', rf_metrics['auc_pr'].values[0], auc_pr if y_test.sum() > 0 else 0),
        ('False Positive Rate', rf_metrics['false_positive_rate'].values[0], fpr),
        ('False Negative Rate', rf_metrics['false_negative_rate'].values[0], fnr),
    ]
    
    for metric_name, rf_val, if_val in metrics_to_compare:
        diff = if_val - rf_val
        diff_str = f"{diff:+.4f}"
        if metric_name in ['False Positive Rate', 'False Negative Rate']:
            # Lower is better for these
            symbol = '✓' if diff < 0 else ('✗' if diff > 0.01 else '~')
        else:
            # Higher is better for these
            symbol = '✓' if diff > 0 else ('✗' if diff < -0.01 else '~')
        print(f"{metric_name:<25} {rf_val:>15.4f} {if_val:>18.4f} {diff_str:>12} {symbol}")
    
    print(f"\n💡 Key Insights:")
    print(f"   ✓ = Isolation Forest better")
    print(f"   ✗ = Random Forest better")
    print(f"   ~ = Similar performance")
else:
    print(f"\n⚠️  Random Forest metrics not found at {rf_metrics_path}")
    print(f"   Cannot perform comparison")

---
## 9. Visualization: Confusion Matrix

In [ ]:
print("\n" + "=" * 80)
print("VISUALIZATION: CONFUSION MATRIX")
print("=" * 80)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Create heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', 
            xticklabels=['Benign', 'Timestomped'],
            yticklabels=['Benign', 'Timestomped'],
            ax=ax, cbar_kws={'label': 'Count'})

ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix - Isolation Forest Timestomping Detection', 
             fontsize=14, fontweight='bold', pad=20)

# Annotations
ax.text(0.5, -0.15, f'False Positives: {fp:,} files ({fpr*100:.3f}% of benign)', 
        ha='center', transform=ax.transAxes, fontsize=10, color='orange')
ax.text(0.5, -0.20, f'False Negatives: {fn:,} files ({fnr*100:.1f}% missed)', 
        ha='center', transform=ax.transAxes, fontsize=10, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
print(f"\n✅ Saved: {OUTPUT_DIR / 'confusion_matrix.png'}")
plt.show()

---
## 10. Visualization: ROC Curve

In [ ]:
print("\n" + "=" * 80)
print("VISUALIZATION: ROC CURVE")
print("=" * 80)

if y_test.sum() > 0:
    fpr_curve, tpr_curve, thresholds_roc = roc_curve(y_test, y_test_pred_proba)
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    ax.plot(fpr_curve, tpr_curve, color='purple', lw=2, 
            label=f'ROC Curve (AUC = {auc_roc:.3f})')
    ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', 
            label='Random Classifier (AUC = 0.500)')
    
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Positive Rate (Recall)', fontsize=12, fontweight='bold')
    ax.set_title('ROC Curve - Isolation Forest Timestomping Detection', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc="lower right", fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'roc_curve.png', dpi=300, bbox_inches='tight')
    print(f"\n✅ Saved: {OUTPUT_DIR / 'roc_curve.png'}")
    plt.show()
else:
    print(f"\n⚠️  Cannot plot ROC curve - no timestomped events in test set")

---
## 11. Visualization: Precision-Recall Curve

In [ ]:
print("\n" + "=" * 80)
print("VISUALIZATION: PRECISION-RECALL CURVE")
print("=" * 80)

if y_test.sum() > 0:
    precision_curve, recall_curve, thresholds_pr = precision_recall_curve(y_test, y_test_pred_proba)
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    ax.plot(recall_curve, precision_curve, color='purple', lw=2,
            label=f'PR Curve (AUC = {auc_pr:.3f})')
    
    baseline = y_test.sum() / len(y_test)
    ax.plot([0, 1], [baseline, baseline], color='navy', lw=2, linestyle='--',
            label=f'Baseline (No Skill = {baseline:.4f})')
    
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('Recall (True Positive Rate)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
    ax.set_title('Precision-Recall Curve - Isolation Forest Timestomping Detection', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc="lower left", fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Add current model point
    ax.plot(test_recall, test_precision, 'ro', markersize=10, 
            label=f'Current Model (P={test_precision:.2f}, R={test_recall:.2f})')
    ax.legend(loc="lower left", fontsize=10)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'precision_recall_curve.png', dpi=300, bbox_inches='tight')
    print(f"\n✅ Saved: {OUTPUT_DIR / 'precision_recall_curve.png'}")
    plt.show()
else:
    print(f"\n⚠️  Cannot plot PR curve - no timestomped events in test set")

---
## 12. Anomaly Score Analysis

Analyze the distribution of anomaly scores for timestomped vs benign files

In [ ]:
print("\n" + "=" * 80)
print("ANOMALY SCORE ANALYSIS")
print("=" * 80)

# Get anomaly scores for test set
anomaly_scores = iso_forest.decision_function(X_test_scaled)

# Create DataFrame for analysis
score_df = pd.DataFrame({
    'anomaly_score': anomaly_scores,
    'is_timestomped': y_test.values
})

print(f"\n📊 Anomaly Score Statistics:")
print(f"\n   Benign Files:")
benign_scores = score_df[score_df['is_timestomped'] == 0]['anomaly_score']
print(f"     Mean: {benign_scores.mean():.4f}")
print(f"     Median: {benign_scores.median():.4f}")
print(f"     Std: {benign_scores.std():.4f}")
print(f"     Range: [{benign_scores.min():.4f}, {benign_scores.max():.4f}]")

print(f"\n   Timestomped Files:")
timestomped_scores = score_df[score_df['is_timestomped'] == 1]['anomaly_score']
print(f"     Mean: {timestomped_scores.mean():.4f}")
print(f"     Median: {timestomped_scores.median():.4f}")
print(f"     Std: {timestomped_scores.std():.4f}")
print(f"     Range: [{timestomped_scores.min():.4f}, {timestomped_scores.max():.4f}]")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Distribution plot
ax1.hist(benign_scores, bins=50, alpha=0.7, label='Benign', color='blue', density=True)
ax1.hist(timestomped_scores, bins=20, alpha=0.7, label='Timestomped', color='red', density=True)
ax1.axvline(benign_scores.mean(), color='blue', linestyle='--', linewidth=2, label=f'Benign Mean ({benign_scores.mean():.3f})')
ax1.axvline(timestomped_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Timestomped Mean ({timestomped_scores.mean():.3f})')
ax1.set_xlabel('Anomaly Score', fontsize=12, fontweight='bold')
ax1.set_ylabel('Density', fontsize=12, fontweight='bold')
ax1.set_title('Distribution of Anomaly Scores', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Box plot
score_df.boxplot(column='anomaly_score', by='is_timestomped', ax=ax2)
ax2.set_xlabel('Is Timestomped', fontsize=12, fontweight='bold')
ax2.set_ylabel('Anomaly Score', fontsize=12, fontweight='bold')
ax2.set_title('Anomaly Score by Class', fontsize=14, fontweight='bold')
ax2.set_xticklabels(['Benign (0)', 'Timestomped (1)'])
plt.suptitle('')  # Remove auto-generated title

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'anomaly_score_distribution.png', dpi=300, bbox_inches='tight')
print(f"\n✅ Saved: {OUTPUT_DIR / 'anomaly_score_distribution.png'}")
plt.show()

# Statistical test
from scipy import stats
statistic, p_value = stats.mannwhitneyu(benign_scores, timestomped_scores, alternative='two-sided')
print(f"\n📈 Mann-Whitney U Test:")
print(f"   Statistic: {statistic:.2f}")
print(f"   P-value: {p_value:.6f}")
print(f"   Significant difference: {'Yes' if p_value < 0.05 else 'No'} (α=0.05)")

---
## 13. Save Model & Results

In [ ]:
print("\n" + "=" * 80)
print("SAVING MODEL & RESULTS")
print("=" * 80)

# Save model
model_path = OUTPUT_DIR / 'isolation_forest_model.joblib'
joblib.dump(iso_forest, model_path)
print(f"\n✅ Saved model: {model_path}")

# Save scaler
scaler_path = OUTPUT_DIR / 'scaler.joblib'
joblib.dump(scaler, scaler_path)
print(f"✅ Saved scaler: {scaler_path}")

# Save evaluation metrics
metrics_dict = {
    'model_type': 'isolation_forest',
    'contamination': contamination_rate,
    'n_estimators': iso_forest.n_estimators,
    'max_samples': iso_forest.max_samples,
    'test_precision': test_precision,
    'test_recall': test_recall,
    'test_f1': test_f1,
    'auc_roc': auc_roc if y_test.sum() > 0 else None,
    'auc_pr': auc_pr if y_test.sum() > 0 else None,
    'true_negatives': int(tn),
    'false_positives': int(fp),
    'false_negatives': int(fn),
    'true_positives': int(tp),
    'false_positive_rate': fpr,
    'false_negative_rate': fnr,
    'test_samples': len(y_test),
    'test_timestomped': int(y_test.sum()),
    'train_samples': len(X_train),
    'train_timestomped': int(y_train.sum())
}

metrics_df = pd.DataFrame([metrics_dict])
metrics_path = OUTPUT_DIR / 'evaluation_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f"✅ Saved metrics: {metrics_path}")

# Save test predictions
predictions_df = pd.DataFrame({
    'y_true': y_test.values,
    'y_pred': y_test_pred,
    'anomaly_score': anomaly_scores,
    'anomaly_proba': y_test_pred_proba
})
predictions_path = OUTPUT_DIR / 'test_predictions.csv'
predictions_df.to_csv(predictions_path, index=False)
print(f"✅ Saved predictions: {predictions_path}")

# Save anomaly score statistics
score_stats_df = score_df.groupby('is_timestomped')['anomaly_score'].describe()
score_stats_path = OUTPUT_DIR / 'anomaly_score_statistics.csv'
score_stats_df.to_csv(score_stats_path)
print(f"✅ Saved anomaly score statistics: {score_stats_path}")

print(f"\n✅ All results saved to: {OUTPUT_DIR}")

---
## 14. Final Summary & Comparison

In [ ]:
print("\n" + "=" * 80)
print("ISOLATION FOREST - FINAL SUMMARY")
print("=" * 80)

print(f"\n🎯 Model Configuration:")
print(f"   Algorithm: Isolation Forest (Unsupervised Anomaly Detection)")
print(f"   Contamination: {contamination_rate:.6f} ({contamination_rate*100:.3f}%)")
print(f"   Trees: {iso_forest.n_estimators}")
print(f"   Max Samples: {iso_forest.max_samples}")
print(f"   Feature Scaling: StandardScaler")

print(f"\n📊 Dataset Summary:")
print(f"   Total records: {len(X):,}")
print(f"   Features: {X.shape[1]}")
print(f"   Train samples: {len(X_train):,}")
print(f"   Test samples: {len(X_test):,}")
print(f"   Class imbalance: 1:{int((y == 0).sum() / y.sum())}")

print(f"\n🎯 Test Set Performance:")
print(f"   Precision: {test_precision:.3f} ({test_precision*100:.1f}%)")
print(f"   Recall:    {test_recall:.3f} ({test_recall*100:.1f}%)")
print(f"   F1-Score:  {test_f1:.3f}")
if y_test.sum() > 0:
    print(f"   AUC-ROC:   {auc_roc:.3f}")
    print(f"   AUC-PR:    {auc_pr:.3f} ⭐ (Primary metric for imbalanced data)")

print(f"\n📈 Confusion Matrix:")
print(f"   TP: {tp:,}  |  FP: {fp:,}")
print(f"   FN: {fn:,}  |  TN: {tn:,}")

print(f"\n💼 Forensic Triage Value:")
print(f"   Files requiring review: {total_flagged:,} out of {len(y_test):,}")
print(f"   Investigation reduction: {(1 - total_flagged/len(y_test))*100:.2f}%")
print(f"   Detection rate: {tp}/{int(y_test.sum())} timestomped files")
print(f"   False positive rate: {fpr*100:.3f}%")
print(f"   False negative rate: {fnr*100:.1f}%")

print(f"\n📁 Saved Artifacts:")
print(f"   Model: isolation_forest_model.joblib")
print(f"   Scaler: scaler.joblib")
print(f"   Metrics: evaluation_metrics.csv")
print(f"   Predictions: test_predictions.csv")
print(f"   Anomaly Statistics: anomaly_score_statistics.csv")
print(f"   Visualizations: confusion_matrix.png, roc_curve.png, precision_recall_curve.png, anomaly_score_distribution.png")

print(f"\n💡 Key Insights:")
if test_f1 > 0.4:
    print(f"   ✓ Isolation Forest shows promising anomaly-based detection")
else:
    print(f"   ⚠️ Isolation Forest underperforms compared to Random Forest")
print(f"   ✓ Unsupervised approach requires no synthetic oversampling (no SMOTE)")
print(f"   ✓ Anomaly scores provide additional forensic insight")

print(f"\n🔄 Comparison Summary:")
if rf_metrics_path.exists():
    print(f"   Random Forest F1-Score: {rf_metrics['test_f1'].values[0]:.3f}")
    print(f"   Isolation Forest F1-Score: {test_f1:.3f}")
    if test_f1 > rf_metrics['test_f1'].values[0]:
        print(f"   🎉 Isolation Forest outperforms Random Forest!")
    elif test_f1 > rf_metrics['test_f1'].values[0] * 0.9:
        print(f"   ✓ Isolation Forest achieves competitive performance")
    else:
        print(f"   ℹ️  Random Forest remains superior for this problem")
        print(f"   💡 Consider ensemble approach combining both models")

print(f"\n⚠️  Limitations & Observations:")
print(f"   - Unsupervised nature may not capture all timestomping patterns")
print(f"   - Sensitive to contamination parameter selection")
print(f"   - Feature scaling required (unlike Random Forest)")
print(f"   - May struggle with timestomping that mimics benign patterns")

print(f"\n🎉 Isolation Forest Analysis Complete!")
print("=" * 80)